In [4]:
# ==============================================================================
# CELL 1: DEPENDENCIES & ENVIRONMENT SETUP
# ==============================================================================
import os
import random
import warnings
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import lightning as pl

from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors
from rdkit.Chem.Scaffolds import MurckoScaffold

from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, r2_score
from lightgbm import LGBMRegressor
from sklearn.multioutput import MultiOutputRegressor

# Chemprop v2 API signatures
from chemprop import data as cpdata, models as cpmodels, featurizers
import chemprop.nn as cpnn

warnings.filterwarnings('ignore')

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)
print("✅ Environment initialized.")

✅ Environment initialized.


In [8]:
# ==============================================================================
# CELL 2: HIGH-PERFORMANCE MOLECULAR FEATURE EXTRACTOR
# ==============================================================================
def calculate_rdkit_features(smiles_list):
    """Generates standard normalized 2D RDKit physical properties."""
    features = []
    desc_list = [d[1] for d in Descriptors._descList]
    
    for smiles in tqdm(smiles_list, desc="Calculating RDKit Descriptors"):
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            features.append([0.0] * len(desc_list))
            continue
        
        if pd.isna(mol):
            features.append([0.0] * len(desc_list))
            continue

        smiles = str(smiles)
        
        row_feats = []
        for desc in desc_list:
            try:
                val = desc(mol)
                if np.isnan(val) or np.isinf(val):
                    row_feats.append(0.0)
                else:
                    row_feats.append(float(val))
            except:
                row_feats.append(0.0)
        features.append(row_feats)
        
    return np.array(features, dtype=np.float32)

def calculate_morgan_fingerprints(smiles_list, radius=2, n_bits=2048):
    """Generates binary topological structural fingerprints."""
    fps = []
    for smiles in tqdm(smiles_list, desc="Generating Morgan Fingerprints"):
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            fps.append([0] * n_bits)
            continue
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
        fps.append(list(fp))
    return np.array(fps, dtype=np.float32)

def get_scaffold(smiles):
    """Calculates Bemis-Murcko scaffolds for grouping constraints."""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None: return "invalid"
        return MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
    except:
        return "invalid"

def calculate_rae(y_true, y_pred):
    """Calculates Challenge Primary Metric: Relative Absolute Error."""
    numerator = np.sum(np.abs(y_true - y_pred))
    denominator = np.sum(np.abs(y_true - np.mean(y_true)))
    return numerator / denominator if denominator != 0 else np.inf

In [10]:
# ==============================================================================
# CELL 3: DATA PIPELINE AND MOE MATRIX HARVESTING
# ==============================================================================
print("Loading datasets from structural pipeline fields...")
train_df = pd.read_csv("../data/MOE_TRAIN.txt")
test_df = pd.read_csv("../data/MOE_TEST.txt")

# Define target labels across primary and efficacy assays
target_cols = [
    'pEC50', 
    'Emax_estimate(log2FCvs.baseline)', 
    'Emax.vs.pos.ctrl_estimate(dimensionless)'
]

train_df['SMILES.smiles'] = train_df['SMILES.smiles'].astype(str)
test_df['SMILES.smiles'] = test_df['SMILES.smiles'].fillna('').astype(str)

# Ensure pristine rows for clean backpropagation profiles
train_df = train_df.dropna(subset=target_cols + ['SMILES.smiles']).reset_index(drop=True)
test_df = test_df.fillna({col: 0.0 for col in target_cols})

# Isolate molecular geometries to prevent feature leaking
moe_drop_cols = target_cols + [
    'MoleculeName', 'SMILES.smiles', 'OCNTBatch', 'Split', 'OCNT_ID', 'source',
    'pEC50_std.error(-log10(molarity))', 'Emax_std.error(log2FCvs.baseline)',
    'Emax.vs.pos.ctrl_std.error(dimensionless)', 'pEC50_ci.lower(-log10(molarity))',
    'pEC50_ci.upper(-log10(molarity))', 'Emax_ci.lower(log2FCvs.baseline)',
    'Emax_ci.upper(log2FCvs.baseline)', 'Emax.vs.pos.ctrl_ci.lower(dimensionless)',
    'Emax.vs.pos.ctrl_ci.upper(dimensionless)'
]
moe_features = [col for col in train_df.columns if col not in moe_drop_cols]

print(f"Parsed {len(moe_features)} explicit Mosaic MOE descriptors.")

# Extract matrices
print("\nExtracting baseline architectures...")
X_train_rdkit = calculate_rdkit_features(train_df['SMILES.smiles'])
X_train_morgan = calculate_morgan_fingerprints(train_df['SMILES.smiles'])
X_train_baseline = np.hstack([X_train_rdkit, X_train_morgan])

X_test_rdkit = calculate_rdkit_features(test_df['SMILES.smiles'])
X_test_morgan = calculate_morgan_fingerprints(test_df['SMILES.smiles'])
X_test_baseline = np.hstack([X_test_rdkit, X_test_morgan])

# Fill static gaps within the MOE properties matrices
X_train_moe = train_df[moe_features].fillna(0.0).values.astype(np.float32)
X_test_moe = test_df[moe_features].fillna(0.0).values.astype(np.float32)

X_train_augmented = np.hstack([X_train_baseline, X_train_moe])
X_test_augmented = np.hstack([X_test_baseline, X_test_moe])

# Setup Scaffold constraints
train_df['scaffold'] = [get_scaffold(s) for s in train_df['SMILES.smiles']]
scaffold_to_group = {s: i for i, s in enumerate(train_df["scaffold"].unique())}
scaffold_groups = train_df["scaffold"].map(scaffold_to_group).values

y_train = train_df[target_cols].values
print(f"\nFinal Engine Vectors Shapes: Baseline={X_train_baseline.shape}, Augmented={X_train_augmented.shape}")

Loading datasets from structural pipeline fields...
Parsed 341 explicit Mosaic MOE descriptors.

Extracting baseline architectures...


Generating Morgan Fingerprints:   0%|          | 0/4139 [00:00<?, ?it/s][07:09:17] DEPRECATION WARNING: please use MorganGenerator
[07:09:17] DEPRECATION WARNING: please use MorganGenerator
[07:09:17] DEPRECATION WARNING: please use MorganGenerator
[07:09:17] DEPRECATION WARNING: please use MorganGenerator
[07:09:17] DEPRECATION WARNING: please use MorganGenerator
[07:09:17] DEPRECATION WARNING: please use MorganGenerator
[07:09:17] DEPRECATION WARNING: please use MorganGenerator
[07:09:17] DEPRECATION WARNING: please use MorganGenerator
[07:09:17] DEPRECATION WARNING: please use MorganGenerator
[07:09:17] DEPRECATION WARNING: please use MorganGenerator
[07:09:17] DEPRECATION WARNING: please use MorganGenerator
[07:09:17] DEPRECATION WARNING: please use MorganGenerator
[07:09:17] DEPRECATION WARNING: please use MorganGenerator
[07:09:17] DEPRECATION WARNING: please use MorganGenerator
[07:09:17] DEPRECATION WARNING: please use MorganGenerator
[07:09:17] DEPRECATION WARNING: please use 


Final Engine Vectors Shapes: Baseline=(4139, 2265), Augmented=(4139, 2606)


In [11]:
# ==============================================================================
# CELL 4: CUSTOM DEEP MULTI-TASK PIPELINE INFRASTRUCTURE
# ==============================================================================
class MultiTaskMPNN(nn.Module):
    def __init__(self, hidden_dim=300, n_tasks_pec50=1, n_tasks_emax=2):
        super().__init__()
        self.message_passing = cpnn.BondMessagePassing(d_h=hidden_dim)
        self.aggregation = cpnn.MeanAggregation()
        
        self.pec50_head = cpnn.RegressionFFN(input_dim=hidden_dim, n_tasks=n_tasks_pec50, hidden_dim=hidden_dim)
        self.emax_head = cpnn.RegressionFFN(input_dim=hidden_dim, n_tasks=n_tasks_emax, hidden_dim=hidden_dim)
        
    def forward(self, batch):
        bmg = batch.bmg  # Resolve internal v2 graph attributes
        h = self.message_passing(bmg)
        embeddings = self.aggregation(h, bmg.batch)
        
        pec50_pred = self.pec50_head(embeddings)
        emax_pred = self.emax_head(embeddings)
        return pec50_pred, emax_pred

In [12]:
# ==============================================================================
# CELL 5: EXPERIMENTAL RUN — LIGHTGBM MULTI-TASK REGRESSION ABLATION
# ==============================================================================
lgbm_params = {
    'n_estimators': 2000,
    'learning_rate': 0.03,
    'num_leaves': 31,
    'max_depth': 6,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.7,
    'verbose': -1,
    'n_jobs': -1
}

def run_lgbm_ablation(X_matrix, y_matrix, groups, label="Experiment"):
    print(f"\nRunning Model Group: {label}")
    gkf = GroupKFold(n_splits=5)
    oof_preds = np.zeros_like(y_matrix)
    fold_raes = []
    
    for fold, (train_idx, val_idx) in enumerate(gkf.split(X_matrix, y_matrix, groups), 1):
        X_tr, X_val = X_matrix[train_idx], X_matrix[val_idx]
        y_tr, y_val = y_matrix[train_idx], y_matrix[val_idx]
        
        model = MultiOutputRegressor(LGBMRegressor(**lgbm_params, random_state=42 + fold))
        model.fit(X_tr, y_tr)
        
        preds = model.predict(X_val)
        oof_preds[val_idx] = preds
        
        rae_p = calculate_rae(y_val[:, 0], preds[:, 0])
        fold_raes.append(rae_p)
        
    cv_rae = np.mean(fold_raes)
    print(f"➔ Overall {label} CV RAE (pEC50): {cv_rae:.4f}")
    return oof_preds, cv_rae

# Run the target configurations
oof_base, cv_base = run_lgbm_ablation(X_train_baseline, y_train, scaffold_groups, "Baseline (RDKit+Morgan)")
oof_aug, cv_aug = run_lgbm_ablation(X_train_augmented, y_train, scaffold_groups, "Augmented (Baseline + Mosaic MOE)")


Running Model Group: Baseline (RDKit+Morgan)
➔ Overall Baseline (RDKit+Morgan) CV RAE (pEC50): 0.5662

Running Model Group: Augmented (Baseline + Mosaic MOE)
➔ Overall Augmented (Baseline + Mosaic MOE) CV RAE (pEC50): 0.5462


In [13]:
# ==============================================================================
# CELL 6: EXPERIMENTAL RUN — DEEP GRAPH MODELING (CHEMPROP MULTI-TASK)
# ==============================================================================
print("\n" + "="*80)
print("RUNNING INTERMEDIATE MULTI-TASK MPNN PIPELINE (DEEP GRAPH ARCHITECTURE)")
print("="*80)

gkf = GroupKFold(n_splits=5)
mpnn_oof_preds = np.zeros((len(train_df), 3))
featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer()

# Balanced Multi-Task Gradient Configuration
W_PEC50, W_EMAX1, W_EMAX2 = 1.0, 0.2, 0.2

for fold, (train_idx, val_idx) in enumerate(gkf.split(train_df, y_train, scaffold_groups), 1):
    print(f"\nTraining Deep MPNN — Fold {fold}/5...")
    
    # Generate points mapping clean structural entities
    tr_points = [cpdata.MoleculeDatapoint(mol=Chem.MolFromSmiles(train_df.iloc[i]['SMILES.smiles']), y=y_train[i].tolist()) for i in train_idx]
    val_points = [cpdata.MoleculeDatapoint(mol=Chem.MolFromSmiles(train_df.iloc[i]['SMILES.smiles']), y=y_train[i].tolist()) for i in val_idx]
    
    train_loader = cpdata.build_dataloader(cpdata.MoleculeDataset(tr_points, featurizer=featurizer), batch_size=64, shuffle=True)
    val_loader = cpdata.build_dataloader(cpdata.MoleculeDataset(val_points, featurizer=featurizer), batch_size=64, shuffle=False)
    
    model = MultiTaskMPNN(hidden_dim=300).cuda() if torch.cuda.is_available() else MultiTaskMPNN(hidden_dim=300)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
    loss_fn = nn.MSELoss()
    
    best_rae = float('inf')
    for epoch in range(30):
        model.train()
        for batch in train_loader:
            if torch.cuda.is_available(): batch = batch.cuda()
            optimizer.zero_grad()
            
            p50_pred, emax_pred = model(batch)
            targets = batch.Y # Extracting uppercase variant confirmed by build tests
            
            loss = (W_PEC50 * loss_fn(p50_pred, targets[:, 0:1]) + 
                    W_EMAX1 * loss_fn(emax_pred[:, 0:1], targets[:, 1:2]) + 
                    W_EMAX2 * loss_fn(emax_pred[:, 1:2], targets[:, 2:3]))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
    # Evaluation Pass
    model.eval()
    fold_preds = []
    with torch.no_grad():
        for batch in val_loader:
            if torch.cuda.is_available(): batch = batch.cuda()
            p50, em = model(batch)
            combined = torch.cat([p50, em], dim=1).cpu().numpy()
            fold_preds.append(combined)
            
    mpnn_oof_preds[val_idx] = np.vstack(fold_preds)
    print(f"Fold {fold} pEC50 RAE: {calculate_rae(y_train[val_idx, 0], mpnn_oof_preds[val_idx, 0]):.4f}")

cv_mpnn = calculate_rae(y_train[:, 0], mpnn_oof_preds[:, 0])
print(f"\n➔ Overall Multi-Task MPNN CV RAE: {cv_mpnn:.4f}")


RUNNING INTERMEDIATE MULTI-TASK MPNN PIPELINE (DEEP GRAPH ARCHITECTURE)

Training Deep MPNN — Fold 1/5...
Fold 1 pEC50 RAE: 0.6318

Training Deep MPNN — Fold 2/5...
Fold 2 pEC50 RAE: 0.6988

Training Deep MPNN — Fold 3/5...
Fold 3 pEC50 RAE: 0.7420

Training Deep MPNN — Fold 4/5...
Fold 4 pEC50 RAE: 0.6870

Training Deep MPNN — Fold 5/5...
Fold 5 pEC50 RAE: 0.7325

➔ Overall Multi-Task MPNN CV RAE: 0.6932


In [16]:
# ==============================================================================
# CELL 7: FINAL ENSEMBLE ORCHESTRATOR & TEST INFERENCE (STRICT COMPLIANCE)
# ==============================================================================
print("\n" + "="*80)
print("ENCODING MODEL DIVERSITY COMBINATIONS")
print("="*80)

final_oof_blend = (0.50 * oof_aug[:, 0]) + (0.50 * mpnn_oof_preds[:, 0])
final_cv_rae = calculate_rae(y_train[:, 0], final_oof_blend)

print(f"⭐ Combined Ensemble Internal Cross-Validation RAE: {final_cv_rae:.4f}")

print("\nGenerating blind submission matrices...")
production_lgbm = MultiOutputRegressor(LGBMRegressor(**lgbm_params, random_state=42))
production_lgbm.fit(X_train_augmented, y_train)
test_lgbm_preds = production_lgbm.predict(X_test_augmented)

# Resolve Name column identity
if 'Molecule Name' in test_df.columns:
    id_col = test_df['Molecule Name']
elif 'MoleculeName' in test_df.columns:
    id_col = test_df['MoleculeName']
elif 'OCNT_ID' in test_df.columns:
    id_col = test_df['OCNT_ID']
else:
    id_col = [f"OADMET-{i:07d}" for i in range(6616, 6616 + len(test_df))]

# Extract original test SMILES
test_smiles = test_df['SMILES.smiles'].values

# --- FIX: Strict Structural and Ordering Alignment ---
submission_df = pd.DataFrame({
    'SMILES': test_smiles,
    'Molecule Name': id_col,
    'pEC50': test_lgbm_preds[:, 0]
})

# Re-verify layout ordering before disk writing
submission_df = submission_df[['SMILES', 'Molecule Name', 'pEC50']]
# -----------------------------------------------------

submission_df.to_csv("submission_day17_ablation.csv", index=False)

print("\n" + "="*80)
print("COMPLIANCE PREVIEW")
print("="*80)
print(submission_df.head(2).to_string(index=False))
print("="*80)
print("✅ Submission successfully formatted and exported to 'submission_day17_ablation.csv'.")


ENCODING MODEL DIVERSITY COMBINATIONS
⭐ Combined Ensemble Internal Cross-Validation RAE: 0.5769

Generating blind submission matrices...

COMPLIANCE PREVIEW
                                           SMILES  Molecule Name    pEC50
S(=O)(=O)(Nc1cc(C(F)(F)F)ccc1)c1cc(C(=O)[O-])ccc1 OADMET-0006617 4.197677
                 Fc1ccc2N(C(=O)c3nnsc3)CCN(C)c2c1 OADMET-0006616 3.742884
✅ Submission successfully formatted and exported to 'submission_day17_ablation.csv'.
